# rlm-gateway on Colab's free GPU

Same root-cause-analysis pipeline as the local project, but running an open-weight model through Ollama on Colab's free T4 GPU instead of your 16GB laptop -- zero API cost, and a bigger model than a laptop could comfortably run.

**Before running anything**: `Runtime` menu -> `Change runtime type` -> Hardware accelerator -> **T4 GPU** -> Save. The check below confirms it's active.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Install and start Ollama

In [ ]:
# Ollama's installer needs zstd to unpack itself, and lspci (from pciutils)
# to detect the T4 GPU and pull in CUDA support -- Colab's base image has
# neither, so without this it silently falls back to CPU-only inference.
!apt-get -qq install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess, time, requests

# Ollama needs to run as a background server process inside this VM.
server = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/version", timeout=3)
        print("Ollama server is up")
        break
    except requests.exceptions.RequestException:
        time.sleep(1)
else:
    raise RuntimeError("Ollama server did not start")

Ollama server is up


In [ ]:
# 14b is a safe first choice for the free T4's 16GB VRAM. If this runs well,
# try swapping to a bigger one, e.g. !ollama pull qwen2.5:32b (may partially
# offload to CPU RAM and run slower, or may not fit at all -- worth testing).
!ollama pull qwen2.5:14b

In [ ]:
# Confirm the model is actually loaded on the GPU, not silently running on
# CPU -- PROCESSOR should say "100% GPU", not "100% CPU".
!ollama run qwen2.5:14b "say hi" --verbose > /dev/null 2>&1
!ollama ps

NAME           ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
qwen2.5:14b    7cdf5a0187d5    9.5 GB    100% GPU     4096       4 minutes from now    


In [ ]:
# Ollama defaults to a small context window (often 2048-4096 tokens). This
# task's tool results can be large, so raise it -- otherwise the model
# silently loses earlier tool-call results once context fills up, which
# looks like confused/repetitive behavior.
%%writefile Modelfile
FROM qwen2.5:14b
PARAMETER num_ctx 16384

Overwriting Modelfile


In [ ]:
!ollama create qwen2.5-14b-16k -f Modelfile
!ollama list


NAME                      ID              SIZE      MODIFIED               
qwen2.5-14b-16k:latest    b54faf739d8b    9.0 GB    Less than a second ago    
qwen2.5:14b               7cdf5a0187d5    9.0 GB    30 seconds ago            


## 2. Install project dependencies

In [ ]:
!pip install -q huggingface-hub pandas pyarrow python-dotenv rlms

## 3. Write out the rlm_gateway package

Same code as `src/rlm_gateway/` in the project -- copied in directly so this notebook is self-contained and doesn't need a GitHub repo or file upload.

In [ ]:
import os
os.makedirs("rlm_gateway", exist_ok=True)
open("rlm_gateway/__init__.py", "w").close()

In [ ]:
%%writefile rlm_gateway/dataset.py
"""
RCAEval dataset access: download, case-id parsing, and per-case loading.

Each case is a folder named "re{suite}{system}_{service}_{fault}_{run}"
containing metrics.parquet (one row per timestamp, one column per
"{service}_{metric}" pair) and inject_time.txt (a single unix timestamp).
The ground-truth root cause is encoded directly in the folder name.

RE1 is metrics-only. RE2/RE3 also include logs.parquet (raw timestamped
log lines: timestamp, container_name, message) and traces.parquet.
"""

import os
import re
from dataclasses import dataclass

import pandas as pd
from huggingface_hub import snapshot_download

DATASET_REPO_ID = "phamquiluan/RCAEval"

# Fault types are spelled out explicitly (rather than matched greedily)
# because service names can themselves contain hyphens, e.g. "ts-auth-service".
CASE_NAME_RE = re.compile(
    r"^re(?P<suite>[123])(?P<system>[a-z]+)_(?P<service>.+)_(?P<fault>cpu|mem|disk|delay|loss)_(?P<run>\d+)$"
)

SYSTEM_NAMES = {"ob": "online-boutique", "ss": "sock-shop", "tt": "train-ticket"}


@dataclass(frozen=True)
class Case:
    case_id: str
    suite: str
    system: str
    root_cause_service: str
    fault_type: str
    run: int


def download_suite(pattern: str = "re1*") -> str:
    """Download a suite of RCAEval cases and return the local snapshot path."""
    return snapshot_download(repo_id=DATASET_REPO_ID, repo_type="dataset", allow_patterns=pattern)


def parse_case_id(case_id: str) -> Case:
    match = CASE_NAME_RE.match(case_id)
    if not match:
        raise ValueError(f"Unrecognized case id: {case_id!r}")
    return Case(
        case_id=case_id,
        suite=f're{match["suite"]}',
        system=SYSTEM_NAMES.get(match["system"], match["system"]),
        root_cause_service=match["service"],
        fault_type=match["fault"],
        run=int(match["run"]),
    )


def list_cases(local_path: str) -> list[Case]:
    return sorted(
        (parse_case_id(name) for name in os.listdir(local_path) if CASE_NAME_RE.match(name)),
        key=lambda c: c.case_id,
    )


def load_case_metrics(local_path: str, case_id: str) -> pd.DataFrame:
    return pd.read_parquet(os.path.join(local_path, case_id, "metrics.parquet"))


def load_case_logs(local_path: str, case_id: str) -> pd.DataFrame:
    """RE2/RE3 only. Columns: timestamp, container_name, message."""
    path = os.path.join(local_path, case_id, "logs.parquet")
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No logs.parquet for {case_id!r}. RE1 cases are metrics-only and have no logs -- "
            f"use download_suite(pattern='re2*') (or 're3*') to get log-bearing cases."
        )
    return pd.read_parquet(path)


def load_inject_time(local_path: str, case_id: str) -> int:
    with open(os.path.join(local_path, case_id, "inject_time.txt")) as f:
        return int(f.read().strip())


Overwriting rlm_gateway/dataset.py


In [ ]:
%%writefile rlm_gateway/tools.py
"""
Custom tools exposed to the RLM root model's REPL.
"""

import pandas as pd

METRIC_SUFFIXES = [
    "_latency-50",
    "_latency-90",
    "_workload",
    "_latency",
    "_load",
    "_error",
    "_cpu",
    "_mem",
]


def split_metric_column(column: str) -> tuple[str, str]:
    for suffix in METRIC_SUFFIXES:
        if column.endswith(suffix):
            return column[: -len(suffix)], suffix[1:]
    raise ValueError(f"Unrecognized metric column: {column!r}")


def list_services(metrics_df: pd.DataFrame) -> list[str]:
    services = {split_metric_column(col)[0] for col in metrics_df.columns if col != "time"}
    return sorted(services)


def list_categories(metrics_df: pd.DataFrame) -> list[str]:
    categories = {split_metric_column(col)[1] for col in metrics_df.columns if col != "time"}
    return sorted(categories)


def _summarize_one_service(metrics_df: pd.DataFrame, service: str, inject_idx: int, window: int) -> str:
    cols = [c for c in metrics_df.columns if c != "time" and split_metric_column(c)[0] == service]
    parts = []
    for col in cols:
        category = split_metric_column(col)[1]
        values = metrics_df[col].tolist()
        pre = values[max(0, inject_idx - window) : inject_idx]
        post = values[inject_idx : inject_idx + window]
        if not pre or not post:
            continue
        pre_mean = sum(pre) / len(pre)
        post_mean = sum(post) / len(post)
        if abs(pre_mean) < 1e-6:
            parts.append(f"{category}: {pre_mean:.4g}->{post_mean:.4g}")
        else:
            change = (post_mean - pre_mean) / abs(pre_mean)
            parts.append(f"{category}: {pre_mean:.4g}->{post_mean:.4g} ({change:+.1%})")
    return ", ".join(parts) if parts else "(no data)"


def summarize_metrics(metrics_df: pd.DataFrame, inject_time: int, window: int = 30) -> str:
    times = metrics_df["time"].tolist()
    inject_idx = min(range(len(times)), key=lambda i: abs(times[i] - inject_time))
    return "\n".join(
        f"{service}: {_summarize_one_service(metrics_df, service, inject_idx, window)}"
        for service in list_services(metrics_df)
    )


def make_service_summary_tool(metrics_df: pd.DataFrame, inject_time: int, window: int = 30):
    """
    Build a get_service_summary(service_name) tool -- returns a plain TEXT
    line comparing that service before vs. after the injection. No raw
    data, no dict shape to guess at, nothing to misread.
    """
    times = metrics_df["time"].tolist()
    inject_idx = min(range(len(times)), key=lambda i: abs(times[i] - inject_time))
    valid_services = set(list_services(metrics_df))

    def get_service_summary(service_name: str) -> str:
        if service_name not in valid_services:
            return f"Unknown service: {service_name!r}. Valid services: {sorted(valid_services)}"
        return _summarize_one_service(metrics_df, service_name, inject_idx, window)

    return get_service_summary


def make_query_service_tool(metrics_df: pd.DataFrame):
    def query_service(service_name: str, category: str | None = None) -> dict:
        matched = [
            col
            for col in metrics_df.columns
            if col != "time"
            and split_metric_column(col)[0] == service_name
            and (category is None or split_metric_column(col)[1] == category)
        ]
        if not matched:
            return {"error": f"No metrics for service={service_name!r} category={category!r}"}
        return metrics_df[["time", *matched]].to_dict(orient="list")

    return query_service


Overwriting rlm_gateway/tools.py


In [ ]:
%%writefile rlm_gateway/_rlm_compat.py
"""
Workaround for a bug in rlms==0.1.3's AnthropicClient: it assumes
response.content[0] is always a text block, but Claude returns a
ThinkingBlock first when extended thinking is on, which has no .text
attribute. Patches completion()/acompletion() to pick out the first
real text block instead. Safe to delete once upstream fixes this.
"""

from rlm.clients import anthropic as rlm_anthropic


def _first_text(content) -> str:
    for block in content:
        if getattr(block, "type", None) == "text":
            return block.text
    return ""


def completion(self, prompt, model=None):
    messages, system = self._prepare_messages(prompt)
    model = model or self.model_name
    if not model:
        raise ValueError("Model name is required for Anthropic client.")
    kwargs = {"model": model, "max_tokens": self.max_tokens, "messages": messages}
    if system:
        kwargs["system"] = system
    response = self.client.messages.create(**kwargs)
    self._track_cost(response, model)
    return _first_text(response.content)


async def acompletion(self, prompt, model=None):
    messages, system = self._prepare_messages(prompt)
    model = model or self.model_name
    if not model:
        raise ValueError("Model name is required for Anthropic client.")
    kwargs = {"model": model, "max_tokens": self.max_tokens, "messages": messages}
    if system:
        kwargs["system"] = system
    response = await self.async_client.messages.create(**kwargs)
    self._track_cost(response, model)
    return _first_text(response.content)


rlm_anthropic.AnthropicClient.completion = completion
rlm_anthropic.AnthropicClient.acompletion = acompletion

Overwriting rlm_gateway/_rlm_compat.py


In [ ]:
%%writefile rlm_gateway/pipeline.py
"""
Builds an RLM instance with query_service/list_services as custom tools,
runs it on one RCAEval case, and checks the prediction against ground truth.
"""

import os

from rlm import RLM
from rlm.clients.anthropic import AnthropicClient
from rlm.clients.openai import OpenAIClient

from . import _rlm_compat  # noqa: F401  (patches AnthropicClient's thinking-block bug)
from .dataset import Case, download_suite, load_case_logs, load_case_metrics, load_inject_time
from .tools import list_categories, list_services, make_query_service_tool, summarize_metrics

PROMPT_TEMPLATE = """\
You are diagnosing a microservice incident. A fault was injected at unix \
timestamp {inject_time} into the "{system}" system, which has these \
services: {services}.

You have two tools:
- list_services() -> list[str]
- query_service(service_name, category=None) -> dict with a "time" key and \
one key per metric column. category is one of: {categories}.

Each query can return hundreds of raw data points. Do NOT print raw \
query_service() results directly -- compute and print a summary instead \
(e.g. mean/min/max before vs. after the injection timestamp).

Use them to inspect metrics around the injection time and figure out which \
service is the root cause of the incident. Respond with ONLY the service \
name as your final answer.
"""


# "local" runs a free, zero-API-cost model via Ollama's OpenAI-compatible
# server (rlm routes backend="vllm" to its generic OpenAI-compatible client,
# which just needs base_url pointed at localhost -- Ollama doesn't check
# the api_key value at all, so any placeholder works).
BACKEND_CONFIGS = {
    "anthropic": {"rlm_backend": "anthropic", "api_key_env": "ANTHROPIC_API_KEY", "default_model": "claude-sonnet-5"},
    "openai": {"rlm_backend": "openai", "api_key_env": "OPENAI_API_KEY", "default_model": "gpt-4o-mini"},
    # Free tier: 50 requests/day on an unfunded OpenRouter account. qwen3-coder
    # is benchmarked as OpenRouter's strongest free option for tool-calling/
    # agentic tasks specifically -- a different model family, not just a
    # bigger download of the same one that struggled locally.
    "openrouter": {
        "rlm_backend": "openrouter",
        "api_key_env": "OPENROUTER_API_KEY",
        "default_model": "qwen/qwen3-coder:free",
        # Normally rlm's own routing fills this in for backend="openrouter",
        # but run_case_direct() calls OpenAIClient directly, bypassing that --
        # so it's set explicitly here to work for both code paths.
        "base_url": "https://openrouter.ai/api/v1",
    },
    "local": {
        "rlm_backend": "vllm",
        "api_key_env": None,
        "default_model": "qwen2.5:7b",
        "base_url": "http://localhost:11434/v1",
    },
}


def run_case(
    local_path: str,
    case: Case,
    backend: str = "anthropic",
    model_name: str | None = None,
    verbose: bool = True,
) -> str:
    metrics_df = load_case_metrics(local_path, case.case_id)
    inject_time = load_inject_time(local_path, case.case_id)
    services = list_services(metrics_df)
    categories = list_categories(metrics_df)

    config = BACKEND_CONFIGS[backend]
    model_name = model_name or config["default_model"]
    api_key = os.getenv(config["api_key_env"]) if config["api_key_env"] else "ollama"

    backend_kwargs = {"model_name": model_name, "api_key": api_key}
    if "base_url" in config:
        backend_kwargs["base_url"] = config["base_url"]

    rlm = RLM(
        backend=config["rlm_backend"],
        backend_kwargs=backend_kwargs,
        environment="local",
        custom_tools={
            "list_services": {
                "tool": lambda: services,
                "description": "List every service name present in this case's metrics.",
            },
            "query_service": {
                "tool": make_query_service_tool(metrics_df),
                "description": (
                    "Fetch metrics for one service, optionally filtered to a "
                    "category (cpu, mem, workload, error, latency-50, latency-90)."
                ),
            },
        },
        verbose=verbose,
    )

    prompt = PROMPT_TEMPLATE.format(
        inject_time=inject_time,
        system=case.system,
        services=", ".join(services),
        categories=", ".join(categories),
    )
    result = rlm.completion(prompt)
    return result.response.strip()


DIRECT_PROMPT_TEMPLATE = """\
A fault was injected into the "{system}" system at unix timestamp {inject_time}.
Below is a summary comparing each service's metrics {window}s before vs. \
{window}s after the injection (mean value and percent change per category):

{summary}

Which service is the root cause of the incident? Respond with ONLY the \
service name, nothing else.
"""


def run_case_direct(
    local_path: str,
    case: Case,
    backend: str = "anthropic",
    model_name: str | None = None,
    window: int = 30,
) -> str:
    """
    Single-shot alternative to run_case(): precomputes the pre/post-injection
    summary ourselves and asks the model one direct question, with no tools
    and no multi-turn REPL loop for it to lose track of. Bypasses RLM
    entirely -- calls the underlying model client straight.
    """
    metrics_df = load_case_metrics(local_path, case.case_id)
    inject_time = load_inject_time(local_path, case.case_id)
    summary = summarize_metrics(metrics_df, inject_time, window=window)

    config = BACKEND_CONFIGS[backend]
    model_name = model_name or config["default_model"]
    api_key = os.getenv(config["api_key_env"]) if config["api_key_env"] else "ollama"

    client_kwargs = {"api_key": api_key, "model_name": model_name}
    if "base_url" in config:
        client_kwargs["base_url"] = config["base_url"]

    client = AnthropicClient(**client_kwargs) if config["rlm_backend"] == "anthropic" else OpenAIClient(**client_kwargs)

    prompt = DIRECT_PROMPT_TEMPLATE.format(
        system=case.system, inject_time=inject_time, summary=summary, window=window
    )
    return client.completion(prompt).strip()


LOGS_ROOT_PROMPT = """\
A fault was injected into the "{system}" system at unix timestamp \
{inject_time}. The context is a LIST of {num_services} strings, one chunk \
per service -- index into it like context[0], context[1], etc. Do NOT \
slice it like a string (e.g. context[:1000] returns whole list items, not \
characters, since there are only {num_services} items total). Each chunk \
covers {window}s before and after the injection.

You have three tools -- use them instead of writing your own \
parsing/counting/ranking code, which is error-prone:
- `log_change_ranking() -> list[dict]`: for EVERY service, exact before/after \
log-line counts around the injection plus "percent_change", already sorted \
by |percent_change| descending (most anomalous first). Each dict also has \
an "index" field -- reuse that directly, do NOT try to re-derive an index \
with context.index(...) (that fails: context holds strings, not these dicts). \
ALWAYS call this FIRST and use "percent_change" to judge anomalies, never \
raw count differences -- a busy gateway service can have a huge raw count \
difference just because it logs the most overall, even with a small/normal \
relative change. A service that drops sharply (large negative percent_change) \
is just as suspicious as one that spikes -- some faults (e.g. high CPU load) \
reduce throughput rather than raising it.
- `service_name_at(index: int) -> str`: the exact service name for \
context[index]. ALWAYS get your final answer from this (or straight from \
log_change_ranking()'s "service" field), never by parsing the \
"=== service: X ===" header text yourself (it's easy to include the \
trailing "===" by mistake).
- `count_logs_in_range(index: int, start_time: float, end_time: float) -> int`: \
exact log-line count for context[index]'s service within \
[start_time, end_time] (unix seconds), for any follow-up drill-down you need \
beyond the standard before/after windows.

IMPORTANT: write all code inside ```repl blocks, never ```python blocks -- \
only ```repl blocks are actually executed. A ```python block is silently \
ignored and does nothing, even if it looks correct.

IMPORTANT: this REPL is NOT a Jupyter notebook. A bare expression on the \
last line (e.g. `my_list[:5]`) produces NO visible output and is silently \
discarded -- you will NOT see the result. You MUST wrap anything you want \
to see in print(...), e.g. `print(my_list[:5])`, every single time, \
including for intermediate values you're using to decide your next step.

After calling log_change_ranking(), also skim the actual message content \
(context[i]) of the top few candidates for a shift in what kind of messages \
they're logging (new error types, stalls, repeated retries) to confirm the \
ranking makes sense. Do NOT just search for the literal words "error" or \
"abnormal" -- a fault often produces no log line containing those words.

You MUST respond with exactly one specific service name, from \
log_change_ranking() or service_name_at(), as your final answer, even if no \
service looks obviously broken -- pick the one that looks most different \
from its own earlier behavior. "No anomaly found" is not an acceptable \
final answer.

Respond with ONLY the service name as your final answer.
"""


def run_case_logs(
    local_path: str,
    case: Case,
    backend: str = "anthropic",
    model_name: str | None = None,
    verbose: bool = True,
    window: int = 15,
    max_iterations: int = 15,
) -> str:
    """
    RE2/RE3 only. Uses RLM the way it's actually designed for: a large,
    unstructured context (raw log lines, chunked per service) passed as
    `prompt`, with the actual question passed separately as `root_prompt`.
    """
    logs_df = load_case_logs(local_path, case.case_id)
    inject_time = load_inject_time(local_path, case.case_id)

    windowed = logs_df[
        (logs_df["timestamp"] >= inject_time - window) & (logs_df["timestamp"] <= inject_time + window)
    ]

    service_names = []
    context_chunks = []
    timestamps_by_service = []
    for container, group in windowed.groupby("container_name"):
        lines = "\n".join(f"{row.timestamp} {row.message}" for row in group.itertuples())
        context_chunks.append(f"=== service: {container} ===\n{lines}")
        service_names.append(container)
        timestamps_by_service.append(group["timestamp"].tolist())

    def service_name_at(index: int) -> str:
        if not (0 <= index < len(service_names)):
            return f"Invalid index {index}. Valid range: 0-{len(service_names) - 1}"
        return service_names[index]

    def count_logs_in_range(index: int, start_time: float, end_time: float) -> int:
        if not (0 <= index < len(timestamps_by_service)):
            return -1
        return sum(1 for t in timestamps_by_service[index] if start_time <= t <= end_time)

    def log_change_ranking() -> list[dict]:
        results = []
        for i, name in enumerate(service_names):
            # Half-open on the injection instant so a log line timestamped
            # exactly at inject_time can't be counted in both windows.
            before = sum(1 for t in timestamps_by_service[i] if inject_time - window <= t < inject_time)
            after = sum(1 for t in timestamps_by_service[i] if inject_time <= t <= inject_time + window)
            pct_change = round((after - before) / before * 100, 1) if before > 0 else (100.0 * after if after > 0 else 0.0)
            results.append(
                {"index": i, "service": name, "before_count": before, "after_count": after, "percent_change": pct_change}
            )
        results.sort(key=lambda r: abs(r["percent_change"]), reverse=True)
        return results

    config = BACKEND_CONFIGS[backend]
    model_name = model_name or config["default_model"]
    api_key = os.getenv(config["api_key_env"]) if config["api_key_env"] else "ollama"

    backend_kwargs = {"model_name": model_name, "api_key": api_key}
    if "base_url" in config:
        backend_kwargs["base_url"] = config["base_url"]

    rlm = RLM(
        backend=config["rlm_backend"],
        backend_kwargs=backend_kwargs,
        environment="local",
        max_iterations=max_iterations,
        # Off by default. Without it, the full growing conversation history
        # gets resent every turn, uncompressed -- this is almost certainly
        # why later iterations got dramatically slower (204s by turn 6).
        compaction=True,
        custom_tools={
            "log_change_ranking": {
                "tool": log_change_ranking,
                "description": (
                    "log_change_ranking() -> list[dict]: before/after log counts and "
                    "percent_change for every service, sorted by |percent_change| "
                    "descending. Call this first; use percent_change, not raw count "
                    "differences, to judge anomalies -- raw differences favor "
                    "whichever service logs the most overall."
                ),
            },
            "service_name_at": {
                "tool": service_name_at,
                "description": (
                    "service_name_at(index: int) -> str: the exact service name for "
                    "a context[] index. Always use this for your final answer "
                    "instead of parsing it out of the chunk's "
                    "'=== service: X ===' header text yourself."
                ),
            },
            "count_logs_in_range": {
                "tool": count_logs_in_range,
                "description": (
                    "count_logs_in_range(index: int, start_time: float, end_time: float) -> int: "
                    "exact log-line count for context[index]'s service within "
                    "[start_time, end_time] (unix seconds). Use this instead of "
                    "writing your own regex/counting code."
                ),
            },
        },
        verbose=verbose,
    )

    root_prompt = LOGS_ROOT_PROMPT.format(
        system=case.system, inject_time=inject_time, window=window, num_services=len(context_chunks)
    )
    result = rlm.completion(context_chunks, root_prompt=root_prompt)
    return result.response.strip()


if __name__ == "__main__":
    import sys

    from dotenv import load_dotenv

    load_dotenv()

    backend = sys.argv[1] if len(sys.argv) > 1 else "anthropic"

    local_path = download_suite()
    case = Case(
        case_id="re1ss_carts_mem_4",
        suite="re1",
        system="sock-shop",
        root_cause_service="carts",
        fault_type="mem",
        run=4,
    )

    prediction = run_case(local_path, case, backend=backend)
    print(f"\nPredicted root cause: {prediction}")
    print(f"Actual root cause:    {case.root_cause_service}")
    print(f"Correct: {case.root_cause_service in prediction}")



Overwriting rlm_gateway/pipeline.py


Note: the default `backend` here is `"local"` (unlike the laptop project, which defaults to `"anthropic"`) since this notebook's whole point is free GPU inference.

## 4. Run one real case (free, uses the Colab GPU)

In [ ]:
import sys
for mod in list(sys.modules):
    if mod.startswith("rlm_gateway"):
        del sys.modules[mod]

from rlm_gateway.dataset import Case, download_suite
from rlm_gateway.pipeline import run_case_logs

local_path = download_suite(pattern="re2ob_checkoutservice_cpu_1/*")
case = Case(case_id="re2ob_checkoutservice_cpu_1", suite="re2", system="online-boutique", root_cause_service="checkoutservice", fault_type="cpu", run=1)

prediction = run_case_logs(local_path, case, backend="local", model_name="qwen2.5-14b-16k")
print(f"\nPredicted: {prediction}")
print(f"Actual:    {case.root_cause_service}")
print(f"Correct:   {case.root_cause_service in prediction}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

╭─ ◆ RLM ━ Recursive Language Model ──────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│    Backend                          vllm                           Environment                     local        │
│    Model                            qwen2.5-14b-16k                Max Iterations                  15           │
│    Max Depth                        1                                                                           │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Context: 2,190 / 108,800 tokens (2% of compaction threshold)

──────────────────────────────────────────────────  Iteration 1  ──────────────────────────────────────────────────

...

╭─ ★ Final Answer ────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  checkoutservice                                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════

  Iterations       4                                                   
                                              Total Time       93.46s                                              
                                              Input Tokens     12,965                                              
                                              Output Tokens    1,009   

═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════


Predicted: checkoutservice
Actual:    checkoutservice
Correct:   True


In [ ]:
import subprocess, time, requests
server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):
    try:
        requests.get("http://localhost:11434/api/version", timeout=3)
        print("Ollama server is up")
        break
    except requests.exceptions.RequestException:
        time.sleep(1)


Ollama server is up


In [ ]:
import sys
for mod in list(sys.modules):
    if mod.startswith("rlm_gateway"):
        del sys.modules[mod]

import random
import time

from rlm_gateway.dataset import download_suite, list_cases
from rlm_gateway.pipeline import run_case_logs

local_path = download_suite(pattern="re2*")
cases = list_cases(local_path)

random.seed(42)
sample = random.sample(cases, min(5, len(cases)))  # start small -- each case takes ~1-3 min

results = []
correct_count = 0
for i, case in enumerate(sample, 1):
    print(f"[{i}/{len(sample)}] {case.case_id} (actual={case.root_cause_service}) ...", flush=True)
    start = time.time()
    try:
        prediction = run_case_logs(
            local_path, case, backend="local", model_name="qwen2.5-14b-16k", verbose=False
        )
        correct = case.root_cause_service in prediction
        error = None
    except Exception as exc:
        prediction = None
        correct = False
        error = str(exc)
    elapsed = time.time() - start
    correct_count += correct
    results.append((case.case_id, case.root_cause_service, prediction, correct, elapsed))
    status = "OK" if error is None else f"ERROR: {error}"
    print(f"  predicted={prediction} correct={correct} ({elapsed:.1f}s) {status if error else ''}")

print(f"\nAccuracy: {correct_count}/{len(sample)} ({correct_count/len(sample):.1%})")
for case_id, actual, predicted, correct, elapsed in results:
    mark = "✓" if correct else "✗"
    print(f"{mark} {case_id}: predicted={predicted} actual={actual} ({elapsed:.1f}s)")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 989 files:   0%|          | 0/989 [00:00<?, ?it/s]

[1/5] re1ob_productcatalogservice_disk_5 (actual=productcatalogservice) ...
  predicted=None correct=False (0.0s) ERROR: [Errno 2] No such file or directory: '/root/.cache/huggingface/hub/datasets--phamquiluan--RCAEval/snapshots/afeacb11bcc94dadfd1c8f483ee4377b2b8b614e/re1ob_productcatalogservice_disk_5/logs.parquet'
[2/5] re1ob_cartservice_cpu_1 (actual=cartservice) ...
  predicted=None correct=False (0.0s) ERROR: [Errno 2] No such file or directory: '/root/.cache/huggingface/hub/datasets--phamquiluan--RCAEval/snapshots/afeacb11bcc94dadfd1c8f483ee4377b2b8b614e/re1ob_cartservice_cpu_1/logs.parquet'
[3/5] re1tt_ts-order-service_delay_2 (actual=ts-order-service) ...
  predicted=None correct=False (0.0s) ERROR: [Errno 2] No such file or directory: '/root/.cache/huggingface/hub/datasets--phamquiluan--RCAEval/snapshots/afeacb11bcc94dadfd1c8f483ee4377b2b8b614e/re1tt_ts-order-service_delay_2/logs.parquet'
[4/5] re1tt_ts-auth-service_cpu_1 (actual=ts-auth-service) ...
  predicted=None correct=